# 팀 통합 균열 및 부식 추가 학습
검수된 팀 자료로 기존 이진 모델 2개를 각각 추가 학습하는 실행 노트북임.

1. 코드 ZIP과 담당자별 데이터 ZIP을 Drive에 저장함.
2. 노션 metadata 형식, 폴리곤과 출처 검수 기록을 검사함.
3. Train과 Validation만으로 학습하고 결과표 확인함.
4. 모델과 설정 확정 후 마지막 셀에서만 Test 실행함.

새 라벨 초안이나 미검토 사진을 자동 승인하지 않음. 이 노트북 자체는 아직 Colab에서 실행되지 않은 상태임.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
BASE = Path('/content/drive/MyDrive/ICT_통합실험0928')
BASE.mkdir(parents=True, exist_ok=True)
# BASE에 팀통합_학습평가0928_코드.zip 저장 필요함.


In [ ]:
import zipfile, subprocess, sys, os, json
code_zip = BASE / '팀통합_학습평가0928_코드.zip'
CODE = Path('/content/crane_team0928')
with zipfile.ZipFile(code_zip) as z:
    for name in z.namelist():
        target = (Path('/content') / name).resolve()
        if not target.is_relative_to(CODE):
            raise ValueError('Unexpected ZIP path: ' + name)
    z.extractall('/content')
os.chdir(CODE)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
import torch
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('런타임 유형을 GPU로 변경 필요함')


## 입력 설정
각 ZIP은 images, labelme, metadata.csv, README.md를 포함하는 검수 완료 제출본이어야 함. Test 파일은 아래 학습 목록에 넣지 않음. 검수 기록 링크는 실제 전달 기록으로 변경 필요함.

In [ ]:
TRAINVAL_ZIPS = {
    'SB': BASE / 'inputs/subin_trainval_20260928_v1.zip',
    'ES': BASE / 'inputs/eunsu_trainval_20260928_v1.zip',
    'HM': BASE / 'inputs/hyemin_trainval_20260928_v1.zip',
}
REVIEW_RECORDS = {'SB':'미입력', 'ES':'미입력', 'HM':'미입력'}
WEIGHTS = {
    'crack': BASE / 'weights/best_E5_tversky.pt',
    'corrosion': BASE / 'weights/best_corrosion_efficientnet-b0.pt',
}
RUN_VERSION = 'v1'
ALLOW_REVIEWED_LOW_RESOLUTION = False  # 승인 기록이 있는 작은 사진을 사용할 때만 True


In [ ]:
from import_metadata import adapt
from data import read_manifests, audit, prepare, save_json

def unpack_metadata(archive, destination):
    destination.mkdir(parents=True, exist_ok=False)
    with zipfile.ZipFile(archive) as z:
        for name in z.namelist():
            if not (destination / name).resolve().is_relative_to(destination.resolve()):
                raise ValueError('Invalid ZIP path')
        z.extractall(destination)
    files = [p for p in destination.rglob('metadata.csv') if '__MACOSX' not in p.parts]
    if len(files) != 1:
        raise ValueError('ZIP마다 metadata.csv 1개 필요함')
    return files[0]

def import_packages(packages, kind):
    for owner in packages:
        if REVIEW_RECORDS.get(owner, '미입력') == '미입력':
            raise ValueError(owner + ': 실제 검수 기록 링크 입력 필요함')
    manifests = []
    for owner, archive in packages.items():
        dest = BASE / f'unpacked_{RUN_VERSION}' / kind / owner
        metadata = unpack_metadata(archive, dest)
        manifest = metadata.parent / 'manifest_v1.json'
        adapt(metadata, manifest, REVIEW_RECORDS[owner])
        manifests.append(manifest)
    return manifests

train_manifests = import_packages(TRAINVAL_ZIPS, 'trainval')
train_rows = read_manifests(train_manifests)
if any(r['split'] not in ('train', 'val') for r in train_rows):
    raise ValueError('학습 ZIP에 Test 또는 미확정 분할이 포함됨')
checked, report = audit(train_rows, allow_reviewed_low_resolution=ALLOW_REVIEWED_LOW_RESOLUTION)
save_json(BASE / f'audit_trainval_{RUN_VERSION}.json', report)
print(json.dumps(report, ensure_ascii=False, indent=2))
if not report['passed']:
    raise ValueError('오류 수정 후 새 버전으로 준비 필요함')
prepared = BASE / f'prepared_trainval_{RUN_VERSION}'
prepare(train_rows, prepared, allow_reviewed_low_resolution=ALLOW_REVIEWED_LOW_RESOLUTION)


## 균열과 부식 추가 학습
두 모델 모두 같은 공통 코드 사용. 기본 가중치와 출력 가중치는 별도 보존함. 실제 사진 Validation IoU로 최고 epoch를 선택하고 정상 오탐을 별도로 검토함.

In [ ]:
from run import train
runs = {}
for task in ('crack', 'corrosion'):
    config = json.loads((CODE / f'templates/{task}_config.json').read_text())
    config['data'] = str(prepared / 'trainval')
    config['weights'] = str(WEIGHTS[task])
    runs[task] = BASE / 'runs' / RUN_VERSION / task
    train(config, runs[task])
    print((runs[task] / '결과표.md').read_text())


## 최종 Test
위 Validation 결과를 검토하고 모델 및 평가 기준을 확정한 뒤 실행 필요함. 마지막 셀은 기본적으로 비활성화 상태임. Test 결과로 재학습하거나 임계값을 조정하지 않음. 실제 Test와 합성 Test 결과는 별도 산출함.

In [ ]:
RUN_FINAL_TEST = False
VALIDATION_DECISION = ''  # 최종 평가 대상 모델로 선택한 근거 기록
TEST_ZIPS = {
    'SB': BASE / 'inputs/subin_test_20260928_v1.zip',
    'ES': BASE / 'inputs/eunsu_test_20260928_v1.zip',
    'HM': BASE / 'inputs/hyemin_test_20260928_v1.zip',
}


In [ ]:
from run import lock, final_test
if not RUN_FINAL_TEST:
    print('최종 Test 미실행. Validation 검토와 자료 확정 후에만 활성화 필요함.')
else:
    if not VALIDATION_DECISION.strip():
        raise ValueError('Validation 검토 근거 입력 필요함')
    test_manifests = import_packages(TEST_ZIPS, 'test')
    test_rows = read_manifests(test_manifests)
    if any(r['split'] != 'test' for r in test_rows):
        raise ValueError('Test ZIP 분할 오류')
    _, report = audit(train_rows + test_rows, allow_reviewed_low_resolution=ALLOW_REVIEWED_LOW_RESOLUTION)
    save_json(BASE / f'audit_all_{RUN_VERSION}.json', report)
    if not report['passed']:
        raise ValueError(json.dumps(report, ensure_ascii=False, indent=2))
    test_prepared = BASE / f'prepared_test_{RUN_VERSION}'
    prepare(test_rows, test_prepared, allow_reviewed_low_resolution=ALLOW_REVIEWED_LOW_RESOLUTION)
    for task in ('crack','corrosion'):
        lock(runs[task], VALIDATION_DECISION)
        out = BASE / 'results' / RUN_VERSION / (task + '_test')
        final_test(runs[task], test_prepared / 'test_sealed', out)
        print((out / '결과표.md').read_text())


Drive의 runs와 results에 실행 설정, 최고 가중치, 점수 JSON, CSV, 결과표 및 비교 그림 저장됨. 기존 실행 폴더를 덮어쓰지 않으며 재실행은 새 RUN_VERSION으로 구분 필요함. Test를 이미 실행한 후에는 버전만 바꾸어 같은 Test에 맞춘 재조정 금지임.